# All Four, Side by Side

The four standalone notebooks ([smile](smile.ipynb), [tribuo](tribuo.ipynb),
[weka](weka.ipynb), [deepnetts](deepnetts.ipynb)) each walk through one library
in detail. This one does something different: it loads **all four at once**,
trains them on the *same* engineered features, and renders a single dashboard so
you can compare them honestly — metrics, per-fixture predictions, and the
tradeoffs that don't show up in a number.

Everything data-related still comes from the shared `.java` pipeline; the only
new thing here is running the four adapters back to back and collecting their
results.

In [1]:
// Keep Smile's SLF4J logging quiet (see the smile notebook for why).
System.setProperty("org.slf4j.simpleLogger.defaultLogLevel", "warn");

In [2]:
%%loadFromPOM
<dependency>
    <groupId>com.fasterxml.jackson.dataformat</groupId>
    <artifactId>jackson-dataformat-csv</artifactId>
    <version>2.17.2</version>
</dependency>
<dependency>
    <groupId>com.github.haifengl</groupId>
    <artifactId>smile-core</artifactId>
    <version>3.1.1</version>
</dependency>
<dependency>
    <groupId>org.tribuo</groupId>
    <artifactId>tribuo-classification-sgd</artifactId>
    <version>4.3.2</version>
</dependency>
<dependency>
    <groupId>nz.ac.waikato.cms.weka</groupId>
    <artifactId>weka-stable</artifactId>
    <version>3.8.6</version>
</dependency>
<dependency>
    <groupId>com.deepnetts</groupId>
    <artifactId>deepnetts-core</artifactId>
    <version>1.13.2</version>
</dependency>
<dependency>
    <groupId>javax.visrec</groupId>
    <artifactId>visrec-api</artifactId>
    <version>1.0.5</version>
</dependency>
<dependency>
    <groupId>org.slf4j</groupId>
    <artifactId>slf4j-simple</artifactId>
    <version>2.0.12</version>
</dependency>

In [3]:
%load shared/Match.java
%load shared/DataLoader.java
%load shared/FormerName.java
%load shared/TeamNames.java
%load shared/EloRating.java
%load shared/RecentForm.java
%load shared/FeatureRow.java
%load shared/FeatureEngineering.java
%load shared/TrainTestSplit.java
%load shared/Metrics.java
%load shared/Predictions.java

## Build the features once

Same pipeline as every notebook. All four libraries train on this exact table.

In [4]:
var all = DataLoader.loadAll("/home/jovyan/data/results.csv");
var names = TeamNames.load("/home/jovyan/data/former_names.csv");
var fe = FeatureEngineering.build(all, names);
var split = TrainTestSplit.chronological(fe.played(), 0.8);

var trainRows = split.train();
var testRows = split.test();
var upcomingRows = fe.upcoming();
int[] testY = TrainTestSplit.toY(testRows);
String[] featureNames = FeatureRow.featureNames();
int numInputs = featureNames.length;

// Result holders, filled in by each library cell below.
var metricsByLib = new java.util.LinkedHashMap<String, Metrics>();
var upcomingProbsByLib = new java.util.LinkedHashMap<String, double[]>();

System.out.println("train=" + trainRows.size() + "  test=" + testRows.size()
    + "  upcoming=" + upcomingRows.size());

train=39546  test=9887  upcoming=44


## Smile

In [5]:
{
    double[][] trainX = TrainTestSplit.toX(trainRows);
    int[] trainYS = TrainTestSplit.toY(trainRows);
    var model = smile.classification.LogisticRegression.fit(trainX, trainYS);

    double[] testProbs = new double[testRows.size()];
    for (int i = 0; i < testRows.size(); i++) {
        double[] post = new double[2];
        model.predict(testRows.get(i).features(), post);
        testProbs[i] = post[1];
    }
    metricsByLib.put("Smile", Metrics.from(testProbs, testY));

    double[] up = new double[upcomingRows.size()];
    for (int i = 0; i < upcomingRows.size(); i++) {
        double[] post = new double[2];
        model.predict(upcomingRows.get(i).features(), post);
        up[i] = post[1];
    }
    upcomingProbsByLib.put("Smile", up);
    System.out.println("Smile:    " + metricsByLib.get("Smile"));
}

Smile:    n=9887  accuracy=0.711  precision=0.680  recall=0.742  f1=0.710  logLoss=0.5561  brier=0.1889


## Tribuo

In [6]:
{
    var labelFactory = new org.tribuo.classification.LabelFactory();
    var trainData = new org.tribuo.MutableDataset<org.tribuo.classification.Label>(
        new org.tribuo.provenance.SimpleDataSourceProvenance("train", labelFactory), labelFactory);
    for (FeatureRow r : trainRows) {
        trainData.add(new org.tribuo.impl.ArrayExample<>(
            new org.tribuo.classification.Label(r.homeWin() ? "1" : "0"), featureNames, r.features()));
    }
    var model = new org.tribuo.classification.sgd.linear.LogisticRegressionTrainer().train(trainData);

    double[] testProbs = new double[testRows.size()];
    for (int i = 0; i < testRows.size(); i++) {
        var ex = new org.tribuo.impl.ArrayExample<>(
            new org.tribuo.classification.Label("1"), featureNames, testRows.get(i).features());
        testProbs[i] = model.predict(ex).getOutputScores().get("1").getScore();
    }
    metricsByLib.put("Tribuo", Metrics.from(testProbs, testY));

    double[] up = new double[upcomingRows.size()];
    for (int i = 0; i < upcomingRows.size(); i++) {
        var ex = new org.tribuo.impl.ArrayExample<>(
            new org.tribuo.classification.Label("1"), featureNames, upcomingRows.get(i).features());
        up[i] = model.predict(ex).getOutputScores().get("1").getScore();
    }
    upcomingProbsByLib.put("Tribuo", up);
    System.out.println("Tribuo:   " + metricsByLib.get("Tribuo"));
}

Tribuo:   n=9887  accuracy=0.705  precision=0.674  recall=0.741  f1=0.706  logLoss=0.5844  brier=0.1957


## Weka

In [7]:
{
    var attributes = new java.util.ArrayList<weka.core.Attribute>();
    for (String f : featureNames) attributes.add(new weka.core.Attribute(f));
    var classValues = new java.util.ArrayList<String>(java.util.List.of("0", "1"));
    attributes.add(new weka.core.Attribute("homeWin", classValues));
    int classIndex = attributes.size() - 1;
    var template = new weka.core.Instances("football", attributes, 0);
    template.setClassIndex(classIndex);

    weka.core.Instances trainData = new weka.core.Instances(template, trainRows.size());
    for (FeatureRow r : trainRows) {
        double[] vals = new double[classIndex + 1];
        double[] f = r.features();
        for (int j = 0; j < f.length; j++) vals[j] = f[j];
        var inst = new weka.core.DenseInstance(1.0, vals);
        inst.setDataset(trainData);
        inst.setClassValue(r.homeWin() ? "1" : "0");
        trainData.add(inst);
    }
    var model = new weka.classifiers.functions.Logistic();
    model.buildClassifier(trainData);
    int homeWinIndex = trainData.classAttribute().indexOfValue("1");

    java.util.function.Function<FeatureRow, Double> prob = r -> {
        try {
            double[] vals = new double[classIndex + 1];
            double[] f = r.features();
            for (int j = 0; j < f.length; j++) vals[j] = f[j];
            var inst = new weka.core.DenseInstance(1.0, vals);
            inst.setDataset(trainData);
            return model.distributionForInstance(inst)[homeWinIndex];
        } catch (Exception e) { throw new RuntimeException(e); }
    };

    double[] testProbs = new double[testRows.size()];
    for (int i = 0; i < testRows.size(); i++) testProbs[i] = prob.apply(testRows.get(i));
    metricsByLib.put("Weka", Metrics.from(testProbs, testY));

    double[] up = new double[upcomingRows.size()];
    for (int i = 0; i < upcomingRows.size(); i++) up[i] = prob.apply(upcomingRows.get(i));
    upcomingProbsByLib.put("Weka", up);
    System.out.println("Weka:     " + metricsByLib.get("Weka"));
}

Weka:     n=9887  accuracy=0.711  precision=0.680  recall=0.742  f1=0.710  logLoss=0.5561  brier=0.1889


## DeepNetts (JSR 381)

DeepNetts needs standardized features (it diverges to `NaN` on raw ones) and
logs every epoch to the console, so this cell standardizes the inputs and
redirects stdout/stderr around training — exactly as in its standalone notebook.

In [8]:
{
    double[] mean = new double[numInputs];
    double[] std = new double[numInputs];
    for (FeatureRow r : trainRows) { double[] f = r.features(); for (int j=0;j<numInputs;j++) mean[j]+=f[j]; }
    for (int j=0;j<numInputs;j++) mean[j]/=trainRows.size();
    for (FeatureRow r : trainRows) { double[] f = r.features(); for (int j=0;j<numInputs;j++) std[j]+=(f[j]-mean[j])*(f[j]-mean[j]); }
    for (int j=0;j<numInputs;j++) { std[j]=Math.sqrt(std[j]/trainRows.size()); if (std[j]==0.0) std[j]=1.0; }

    java.util.function.Function<double[], float[]> scale = f -> {
        float[] out = new float[numInputs];
        for (int j=0;j<numInputs;j++) out[j]=(float)((f[j]-mean[j])/std[j]);
        return out;
    };

    String[] columnNames = new String[numInputs + 1];
    System.arraycopy(featureNames, 0, columnNames, 0, numInputs);
    columnNames[numInputs] = "homeWin";
    var trainSet = new deepnetts.data.TabularDataSet(numInputs, 1);
    trainSet.setColumnNames(columnNames);
    for (FeatureRow r : trainRows) {
        float label = (r.homeWin() != null && r.homeWin()) ? 1f : 0f;
        trainSet.add(new deepnetts.data.TabularDataSet.Item(scale.apply(r.features()), new float[]{label}));
    }

    var realOut = System.out; var realErr = System.err;
    var devNull = new java.io.PrintStream(java.io.OutputStream.nullOutputStream());
    deepnetts.net.FeedForwardNetwork net;
    System.setOut(devNull); System.setErr(devNull);
    try {
        net = deepnetts.net.FeedForwardNetwork.builder()
            .addInputLayer(numInputs)
            .addOutputLayer(1, deepnetts.net.layers.activation.ActivationType.SIGMOID)
            .lossFunction(deepnetts.net.loss.LossType.CROSS_ENTROPY)
            .build();
        var trainer = net.getTrainer();
        trainer.setMaxError(0.01f); trainer.setMaxEpochs(30); trainer.setLearningRate(0.1f);
        trainer.train(trainSet);
    } finally { System.setOut(realOut); System.setErr(realErr); }

    java.util.function.Function<FeatureRow, Double> prob = r -> {
        net.setInput(deepnetts.util.Tensor.create(1, numInputs, scale.apply(r.features())));
        return (double) net.getOutput()[0];
    };

    double[] testProbs = new double[testRows.size()];
    for (int i = 0; i < testRows.size(); i++) testProbs[i] = prob.apply(testRows.get(i));
    metricsByLib.put("DeepNetts", Metrics.from(testProbs, testY));

    double[] up = new double[upcomingRows.size()];
    for (int i = 0; i < upcomingRows.size(); i++) up[i] = prob.apply(upcomingRows.get(i));
    upcomingProbsByLib.put("DeepNetts", up);
    System.out.println("DeepNetts: " + metricsByLib.get("DeepNetts"));
}

DeepNetts: n=9887  accuracy=0.706  precision=0.669  recall=0.756  f1=0.710  logLoss=0.5752  brier=0.1939


## Dashboard 1 — the scoreboard

All four, scored through the same `Metrics`, on the same held-out test set. The
best value in each column is highlighted.

In [9]:
// Build an HTML metrics table; bold the best cell per column.
// accuracy/precision/recall/f1: higher is better. logLoss/brier: lower is better.
String[] libs = metricsByLib.keySet().toArray(new String[0]);

double bestAcc = metricsByLib.values().stream().mapToDouble(m -> m.accuracy).max().getAsDouble();
double bestF1  = metricsByLib.values().stream().mapToDouble(m -> m.f1).max().getAsDouble();
double bestLog = metricsByLib.values().stream().mapToDouble(m -> m.logLoss).min().getAsDouble();
double bestBri = metricsByLib.values().stream().mapToDouble(m -> m.brier).min().getAsDouble();

java.util.function.BiFunction<Double, Double, String> cell = (val, best) -> {
    boolean isBest = Math.abs(val - best) < 1e-9;
    String s = String.format("%.4f", val);
    return isBest ? "<td style='background:#e8f5e9;font-weight:bold'>" + s + "</td>" : "<td>" + s + "</td>";
};

var sb = new StringBuilder();
sb.append("<table style='border-collapse:collapse' border='1' cellpadding='6'>");
sb.append("<tr style='background:#263238;color:white'><th>library</th><th>accuracy</th>")
  .append("<th>precision</th><th>recall</th><th>F1</th><th>log-loss</th><th>Brier</th></tr>");
for (String lib : libs) {
    Metrics m = metricsByLib.get(lib);
    sb.append("<tr><td style='font-weight:bold'>").append(lib).append("</td>");
    sb.append(cell.apply(m.accuracy, bestAcc));
    sb.append(String.format("<td>%.4f</td>", m.precision));
    sb.append(String.format("<td>%.4f</td>", m.recall));
    sb.append(cell.apply(m.f1, bestF1));
    sb.append(cell.apply(m.logLoss, bestLog));
    sb.append(cell.apply(m.brier, bestBri));
    sb.append("</tr>");
}
sb.append("</table>");
sb.append("<p style='color:#555'>Green = best in column. Higher is better for accuracy/precision/recall/F1; lower for log-loss/Brier.</p>");
display(sb.toString(), "text/html");
System.out.flush();

library,accuracy,precision,recall,F1,log-loss,Brier
Smile,0.7108,0.6802,0.7424,0.7100,0.5561,0.1889
Tribuo,0.7055,0.6737,0.7409,0.7057,0.5844,0.1957
Weka,0.7108,0.6802,0.7424,0.7100,0.5561,0.1889
DeepNetts,0.7058,0.6695,0.7560,0.7101,0.5752,0.1939


## Dashboard 2 — same fixture, four opinions

The 2026 World Cup group-stage fixtures, with each library's P(home win) side by
side. Where the four agree, you can trust the call; where they diverge, you're
seeing the optimizer differences from the scoreboard play out match by match.

In [10]:
var sb2 = new StringBuilder();
sb2.append("<table style='border-collapse:collapse' border='1' cellpadding='5'>");
sb2.append("<tr style='background:#263238;color:white'><th>date</th><th>home</th><th>away</th>");
for (String lib : libs) sb2.append("<th>").append(lib).append("</th>");
sb2.append("<th>spread</th></tr>");

for (int i = 0; i < upcomingRows.size(); i++) {
    FeatureRow f = upcomingRows.get(i);
    double min = 1.0, max = 0.0;
    for (String lib : libs) { double p = upcomingProbsByLib.get(lib)[i]; min = Math.min(min, p); max = Math.max(max, p); }
    double spread = max - min;
    // shade rows where the libraries disagree the most
    String rowStyle = spread > 0.15 ? " style='background:#fff3e0'" : "";
    sb2.append("<tr").append(rowStyle).append(">");
    sb2.append("<td>").append(f.date()).append("</td>");
    sb2.append("<td>").append(f.homeTeam()).append("</td>");
    sb2.append("<td>").append(f.awayTeam()).append("</td>");
    for (String lib : libs) {
        double p = upcomingProbsByLib.get(lib)[i];
        sb2.append(String.format("<td style='text-align:right'>%.0f%%</td>", p * 100));
    }
    sb2.append(String.format("<td style='text-align:right;color:#888'>%.0f%%</td>", spread * 100));
    sb2.append("</tr>");
}
sb2.append("</table>");
sb2.append("<p style='color:#555'>Orange rows: the libraries disagree by more than 15 points on P(home win).</p>");
display(sb2.toString(), "text/html");
System.out.flush();

date,home,away,Smile,Tribuo,Weka,DeepNetts,spread
2026-06-19,Scotland,Morocco,21%,9%,21%,12%,11%
2026-06-19,Brazil,Haiti,83%,94%,83%,89%,11%
2026-06-19,United States,Australia,50%,51%,50%,52%,2%
2026-06-19,Turkey,Paraguay,49%,49%,49%,48%,1%
2026-06-20,Germany,Ivory Coast,64%,68%,64%,69%,5%
2026-06-20,Ecuador,Curaçao,88%,97%,88%,96%,8%
2026-06-20,Netherlands,Sweden,61%,72%,61%,67%,10%
2026-06-20,Tunisia,Japan,13%,6%,13%,7%,7%
2026-06-21,Belgium,Iran,47%,43%,47%,44%,5%
2026-06-21,New Zealand,Egypt,24%,15%,24%,16%,8%


## The honest comparison

**On accuracy, it's a wash.** Four mature libraries, the same six features, the
same chronological split — and the test-set accuracy spans about a single point.
Smile and Weka agree to every digit we print (both use a direct quasi-Newton solver),
while Tribuo and DeepNetts trail by a hair because they optimize with SGD. If all
you care about is "how often is it right," the choice doesn't matter.

**The differences that matter aren't in the numbers.** They're in everything
around them:

- **The adapter.** Smile took a `double[][]`; Tribuo wanted typed
  `Example<Label>` objects; Weka wanted an `Instances` table with a nominal
  class; DeepNetts wanted a `TabularDataSet` of `float[]`. The shape of the data
  each library demands is the single biggest practical difference between them.
- **What you get back.** Tribuo records full provenance automatically — an audit
  trail you didn't ask for. Weka prints a coefficient-and-odds-ratio table from
  `toString()`. Smile hands you the raw coefficients. DeepNetts gives you a
  network you can grow into a deeper model.
- **How much care it needs.** DeepNetts was the only one that *required* feature
  standardization (raw inputs diverged to `NaN`) and had a logging/eval rough
  edge. The other three tolerated raw, unscaled features without complaint.
- **The coefficients lie either way.** Whichever library you read them from, our
  unscaled, collinear features produce coefficients that aren't safe to
  interpret — Smile and Weka even print mirror-image signs for the identical
  model. Accuracy and interpretability are separate properties.

**So there's no winner — there's a fit.** Tribuo for regulated production where
provenance and type safety earn their keep; Smile when you want the least
ceremony; Weka for its mature catalog and built-in reporting; DeepNetts when
coding to the JSR 381 standard matters or logistic regression is step one toward
a neural net. The one thing they share is the thing we started with: every one of
them trains and serves in the JVM, right next to your data.